# SA/2D Connection Authoring

Create, inspect, delete, and re-create SA/2D area connections in a real HEC-RAS
geometry file. This notebook demonstrates the full CRUD cycle using
`GeomLateral` connection authoring methods:

- `get_connections()` — list all connections
- `get_connection_line_coords()` — read polyline XY coordinates
- `get_connection_profile()` — read weir crest station/elevation profile
- `get_connection_gates()` — read gate definitions
- `delete_connection()` — remove a connection block
- `set_connection()` — create or replace a connection block
- `set_connection_profile()` — write weir crest profile
- `set_connection_gates()` — write gate definitions

## Development Mode

In [ ]:
#!pip install --upgrade ras-commander

In [ ]:
# =============================================================================
# DEVELOPMENT MODE TOGGLE
# =============================================================================
USE_LOCAL_SOURCE = False

if USE_LOCAL_SOURCE:
    import sys
    from pathlib import Path

    cwd = Path.cwd()
    local_path = cwd if (cwd / "ras_commander").exists() else cwd.parent
    if str(local_path) not in sys.path:
        sys.path.insert(0, str(local_path))
    print(f"LOCAL SOURCE MODE: loading from {local_path / 'ras_commander'}")
else:
    print("PIP PACKAGE MODE: loading installed ras-commander")

from pathlib import Path
import logging
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from ras_commander import RasExamples
from ras_commander.geom import GeomLateral

warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("ras_commander").setLevel(logging.CRITICAL)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

import ras_commander

print(f"Loaded: {ras_commander.__file__}")

## Parameters

In [ ]:
PROJECT_NAME = "BaldEagleCrkMulti2D"
PROJECT_SUFFIX = "214_connection_authoring"
GEOM_FILE_NAME = "BaldEagleDamBrk.g13"
TARGET_CONNECTION = "Dam"

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / "ras_commander").exists() else cwd.parent
WORK_ROOT = REPO_ROOT / "working" / "connection_authoring"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Working folder: {WORK_ROOT}")

## Extract And Inspect The Project

In [ ]:
project_path = RasExamples.extract_project(
    PROJECT_NAME,
    output_path=WORK_ROOT,
    suffix=PROJECT_SUFFIX,
)
geom_file = project_path / GEOM_FILE_NAME

print(f"Project path: {project_path}")
print(f"Geometry file: {geom_file.name}")

connections = GeomLateral.get_connections(geom_file)
display_cols = ["Name", "Type", "From", "To", "NumPoints", "LinePoints", "HasGate"]
display(connections[display_cols])

print(f"\nFound {len(connections)} connections")

## Inspect The Target Connection

Read the "Dam" connection's polyline coordinates, weir crest profile, and gate
definitions. These values are saved so the connection can be re-created after
deletion.

In [ ]:
original_coords = GeomLateral.get_connection_line_coords(geom_file, TARGET_CONNECTION)
original_profile = GeomLateral.get_connection_profile(geom_file, TARGET_CONNECTION)
original_gates = GeomLateral.get_connection_gates(geom_file, TARGET_CONNECTION)

print(f"Connection line: {len(original_coords)} points")
display(original_coords)

print(f"\nWeir crest profile: {len(original_profile)} station/elevation pairs")
display(original_profile)

print(f"\nGates: {len(original_gates)}")
gate_cols = ["GateName", "Width", "Height", "InvertElevation", "GateCoefficient", "NumOpenings", "OpeningStations"]
display(original_gates[gate_cols])

## Plot The Original Connection Line

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(original_coords["X"], original_coords["Y"], "o-", linewidth=2, label="Original Dam")
ax.set_title(f"Connection Line: {TARGET_CONNECTION}")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## Delete The Connection

Remove the "Dam" connection entirely and confirm it is gone.

In [ ]:
GeomLateral.delete_connection(geom_file, TARGET_CONNECTION, create_backup=True)

after_delete = GeomLateral.get_connections(geom_file)
display(after_delete[["Name", "Type", "From", "To"]])

assert TARGET_CONNECTION not in after_delete["Name"].values
assert len(after_delete) == len(connections) - 1
print(f"Deleted '{TARGET_CONNECTION}'. {len(after_delete)} connections remain.")

## Re-Create The Connection With Modifications

Re-add the "Dam" connection using the original polyline coordinates but with
modified hydraulic parameters:

- Weir coefficient: 3.82 &rarr; **2.60** (broad-crested weir instead of ogee)
- Weir width: 100 &rarr; **120** ft
- Overflow method: remains 2D

In [ ]:
coords_tuples = list(zip(
    original_coords["X"].tolist(),
    original_coords["Y"].tolist(),
))

dam_row = connections.loc[connections["Name"] == TARGET_CONNECTION].iloc[0]

GeomLateral.set_connection(
    geom_file,
    TARGET_CONNECTION,
    coords_tuples,
    upstream_area=dam_row["From"],
    downstream_area=dam_row["To"],
    routing_type=1,
    weir_width=120.0,
    weir_coef=2.6,
    overflow_method_2d=True,
    create_backup=False,
)

after_recreate = GeomLateral.get_connections(geom_file)
display(after_recreate[["Name", "Type", "From", "To", "NumPoints", "LinePoints"]])

assert TARGET_CONNECTION in after_recreate["Name"].values
assert len(after_recreate) == len(connections)
print(f"Re-created '{TARGET_CONNECTION}' with modified parameters.")

## Restore The Weir Crest Profile

The default `set_connection()` writes a flat 2-point placeholder profile.
Replace it with the original weir crest, but raise all elevations by 0.5 ft
to simulate a design modification.

In [ ]:
modified_profile = original_profile.copy()
modified_profile["Elevation"] = modified_profile["Elevation"] + 0.5

GeomLateral.set_connection_profile(
    geom_file,
    TARGET_CONNECTION,
    modified_profile,
    create_backup=False,
)

restored_profile = GeomLateral.get_connection_profile(geom_file, TARGET_CONNECTION)

print(f"Original profile points: {len(original_profile)}")
print(f"Restored profile points: {len(restored_profile)}")

comparison = pd.DataFrame({
    "Station": original_profile["Station"],
    "Original Elev": original_profile["Elevation"],
    "Modified Elev": restored_profile["Elevation"],
    "Delta": restored_profile["Elevation"] - original_profile["Elevation"],
})
display(comparison)

assert all(comparison["Delta"].round(3) == 0.5)
print("Profile restored with +0.5 ft elevation offset.")

## Add Modified Gates

Re-add a gate based on the original, but widen it from 7 ft to **10 ft** and
add a second gate at a different station.

In [ ]:
new_gates = [
    {
        "GateName": "Main Gate",
        "Width": 10.0,
        "Height": 15.0,
        "InvertElevation": 590.0,
        "GateCoefficient": 0.65,
        "NumOpenings": 2,
        "OpeningStations": [5745.0, 5765.0],
    },
    {
        "GateName": "Auxiliary Gate",
        "Width": 5.0,
        "Height": 8.0,
        "InvertElevation": 595.0,
        "GateCoefficient": 0.60,
        "NumOpenings": 1,
        "OpeningStations": [3500.0],
    },
]

GeomLateral.set_connection_gates(
    geom_file,
    TARGET_CONNECTION,
    new_gates,
    create_backup=False,
)

restored_gates = GeomLateral.get_connection_gates(geom_file, TARGET_CONNECTION)
display(restored_gates[gate_cols])

assert len(restored_gates) == 2
assert restored_gates.iloc[0]["GateName"] == "Main Gate"
assert restored_gates.iloc[0]["Width"] == 10.0
assert restored_gates.iloc[1]["GateName"] == "Auxiliary Gate"
print(f"Wrote {len(restored_gates)} gates to '{TARGET_CONNECTION}'.")

## Compare Original And Modified Connection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Connection line comparison
ax = axes[0]
new_coords = GeomLateral.get_connection_line_coords(geom_file, TARGET_CONNECTION)
ax.plot(original_coords["X"], original_coords["Y"], "o-", linewidth=2, label="Original", alpha=0.6)
ax.plot(new_coords["X"], new_coords["Y"], "s--", linewidth=2, label="Re-created", alpha=0.8)
ax.set_title("Connection Line")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)
ax.legend()

# Profile comparison
ax = axes[1]
ax.plot(
    original_profile["Station"], original_profile["Elevation"],
    "o-", linewidth=2, label="Original", alpha=0.6,
)
ax.plot(
    restored_profile["Station"], restored_profile["Elevation"],
    "s--", linewidth=2, label="Modified (+0.5 ft)", alpha=0.8,
)
ax.set_title("Weir Crest Profile")
ax.set_xlabel("Station")
ax.set_ylabel("Elevation")
ax.grid(True, alpha=0.25)
ax.legend()

fig.suptitle(f"Connection: {TARGET_CONNECTION}", fontsize=13)
fig.tight_layout()
plt.show()

## Summary

In [ ]:
summary = pd.DataFrame([{
    "Connection": TARGET_CONNECTION,
    "Original Weir Coef": 3.82,
    "Modified Weir Coef": 2.60,
    "Original Weir Width": 100,
    "Modified Weir Width": 120,
    "Profile Elevation Shift": "+0.5 ft",
    "Original Gates": len(original_gates),
    "Modified Gates": len(restored_gates),
    "Line Points Preserved": len(new_coords) == len(original_coords),
    "Profile Points Preserved": len(restored_profile) == len(original_profile),
}])
display(summary)
print("Connection authoring workflow complete.")